# 02 · Jalankan Dashboard Streamlit dari Colab

Menampilkan model hasil notebook **01** sebagai dashboard Streamlit (6 halaman). Tidak ada pelatihan ulang.

1. Jalankan notebook **01** dulu di sesi Colab yang sama (atau seret `hasil_model.zip` ke panel 📁 Files).
2. **Runtime → Run all** (±1 menit).
3. Klik link `https://….trycloudflare.com` di sel terakhir. Link aktif selama sesi Colab berjalan.

## 1 · Siapkan aplikasi + ambil hasil model

In [ ]:
#@title Siapkan aplikasi Streamlit (klik ▶) { display-mode: "form" }
import base64, io, os, shutil, tarfile, zipfile, glob
APP = "/content/ruang_sales_app"
PAYLOAD = "..."
shutil.rmtree(APP, ignore_errors=True); os.makedirs(APP)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as tar:
    tar.extractall(APP)
del PAYLOAD

# Ambil hasil model dari notebook 01 (folder di sesi ini, atau hasil_model.zip di panel Files)
sumber = None
if os.path.isfile("/content/ruang_sales_model/hasil_model/model_results.json"):
    shutil.copytree("/content/ruang_sales_model/hasil_model", f"{APP}/hasil_model"); sumber = "folder hasil notebook 01 (sesi ini)"
else:
    for z in sorted(glob.glob("/content/hasil_model*.zip")):
        with zipfile.ZipFile(z) as arc:
            if "hasil_model/model_results.json" in arc.namelist():
                arc.extractall(APP); sumber = os.path.basename(z); break
if sumber:
    print("✅ Hasil model:", sumber)
else:
    print("⚠️ Hasil model belum ada. Jalankan notebook 01 dulu, atau seret hasil_model.zip ke panel Files lalu jalankan sel ini lagi.")
    print("   (Dashboard tetap bisa dibuka; halaman awalnya meminta unggah hasil_model.zip.)")


## 2 · Pasang Streamlit

In [ ]:
!pip install -q "streamlit>=1.55,<2.0" "plotly>=6.0,<8.0"

## 3 · Nyalakan Streamlit

In [ ]:
import subprocess, sys, time, urllib.request
subprocess.run(["pkill", "-f", "streamlit run"])
log = open("/content/streamlit.log", "w")
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0",
                  "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                 cwd=APP, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read() == b"ok":
            print("✅ Streamlit menyala"); break
    except Exception:
        time.sleep(1)
else:
    print("❌ Streamlit belum menyala:"); print(open("/content/streamlit.log").read()[-3000:])


## 4 · Buka dashboard (link publik)

In [ ]:
import re
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if found:
        url = found[0]; break
print("🌐 Dashboard:", url or "belum dapat link — jalankan sel ini lagi")


### Lain-lain

In [ ]:
print(open("/content/streamlit.log").read()[-4000:])  # lihat log bila ada error

In [ ]:
subprocess.run(["pkill", "-f", "cloudflared tunnel"]); subprocess.run(["pkill", "-f", "streamlit run"]); print("Dimatikan")